# Genetic Algorithms - Tutorial 4: Real-World Applications

Welcome to the final tutorial! This tutorial demonstrates how to apply GAs to real-world problems.

## Applications Covered:

1. **Hyperparameter Optimization** - Tuning ML models
2. **Feature Selection** - Finding optimal feature subsets
3. **Job Scheduling** - Resource allocation
4. **Portfolio Optimization** - Finance applications

**Prerequisites:** Tutorials 1-3

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

# Import all utilities
sys.path.append('../GA_Tutorial_1_Basics')
sys.path.append('../GA_Tutorial_2_Intermediate')

from ga_utils_basics import *
from ga_utils_intermediate import *
from ga_utils_applications import *

# For ML examples
try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.tree import DecisionTreeClassifier
    sklearn_available = True
except:
    sklearn_available = False
    print("⚠️ scikit-learn not available - some examples will be limited")

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful!")

## Application 1: Hyperparameter Optimization

### Problem

Machine learning models have **hyperparameters** that must be tuned:
- Learning rate
- Number of estimators
- Max depth
- Regularization strength

### Traditional Approaches

- ❌ **Grid Search**: Exponentially expensive
- ❌ **Random Search**: May miss optimal regions
- ✅ **Genetic Algorithm**: Intelligent exploration!

### GA Encoding

```
Chromosome: [lr_bits, n_estimators_bits, max_depth_bits, ...]
Decode to: {'learning_rate': 0.01, 'n_estimators': 100, 'max_depth': 10}
```

In [ ]:
if sklearn_available:
    print("="*70)
    print("Hyperparameter Optimization for Random Forest")
    print("="*70)

    # Generate synthetic data
    X_train, y_train, X_val, y_val = generate_synthetic_ml_data(
        n_samples=500, n_features=20, n_informative=10
    )

    print(f"\nData: {X_train.shape[0]} training samples, {X_val.shape[0]} validation samples")
    print(f"Features: {X_train.shape[1]}")

    # Define hyperparameter ranges
    param_ranges = {
        'n_estimators': (10, 200),
        'max_depth': (2, 20),
        'min_samples_split': (2, 20)
    }

    print(f"\nOptimizing {len(param_ranges)} hyperparameters:")
    for param, (min_val, max_val) in param_ranges.items():
        print(f"  {param}: [{min_val}, {max_val}]")

    # Simple hyperparameter optimization GA
    def hyperparameter_ga(param_ranges, X_train, y_train, X_val, y_val,
                         pop_size=20, max_generations=30):
        """
        Optimize hyperparameters using GA.
        """
        # Initialize population (real encoding)
        n_params = len(param_ranges)
        population = initialize_population_real(pop_size, n_params, (0, 1))

        history = {'best_score': [], 'best_params': []}

        for gen in range(max_generations):
            # Evaluate fitness
            fitness_values = []
            for individual in population:
                # Decode to hyperparameters
                hyperparams = {}
                for i, (param_name, (min_val, max_val)) in enumerate(param_ranges.items()):
                    value = min_val + individual[i] * (max_val - min_val)
                    if param_name != 'min_samples_split':  # Keep as int
                        hyperparams[param_name] = int(value)
                    else:
                        hyperparams[param_name] = int(value)

                # Evaluate
                score = evaluate_ml_model(hyperparams, X_train, y_train,
                                         X_val, y_val, RandomForestClassifier)
                fitness_values.append(score)

            fitness_values = np.array(fitness_values)

            # Track best
            best_idx = np.argmax(fitness_values)
            best_individual = population[best_idx]
            best_params = {}
            for i, (param_name, (min_val, max_val)) in enumerate(param_ranges.items()):
                value = min_val + best_individual[i] * (max_val - min_val)
                best_params[param_name] = int(value)

            history['best_score'].append(np.max(fitness_values))
            history['best_params'].append(best_params)

            if gen % 10 == 0 or gen == max_generations - 1:
                print(f"Gen {gen:2d} | Best score: {np.max(fitness_values):.4f} | Params: {best_params}")

            # Selection, crossover, mutation
            parents = rank_based_selection(population, fitness_values, pop_size)

            offspring = []
            for i in range(0, pop_size, 2):
                p1, p2 = parents[i], parents[min(i+1, pop_size-1)]
                c1, c2 = simulated_binary_crossover(p1, p2, eta=20, bounds=(0, 1))
                c1 = polynomial_mutation(c1, 0.1, (0, 1), eta=20)
                c2 = polynomial_mutation(c2, 0.1, (0, 1), eta=20)
                offspring.extend([c1, c2])

            population = np.array(offspring[:pop_size])

        return best_params, history

    # Run optimization
    print("\n" + "-"*70)
    print("Running GA Hyperparameter Optimization...")
    print("-"*70 + "\n")

    best_params, history = hyperparameter_ga(
        param_ranges, X_train, y_train, X_val, y_val,
        pop_size=20, max_generations=30
    )

    print(f"\n{'='*70}")
    print("FINAL RESULT")
    print(f"{'='*70}")
    print(f"Best hyperparameters: {best_params}")
    print(f"Best validation score: {history['best_score'][-1]:.4f}")

else:
    print("⚠️ Skipping hyperparameter optimization (scikit-learn required)")

## Application 2: Feature Selection

### Problem

With many features:
- ❌ Overfitting
- ❌ Slow training
- ❌ Poor interpretability

**Goal**: Find minimal feature subset with maximum accuracy.

### GA Encoding (Binary)

```
Chromosome: [1, 0, 1, 0, 0, 1, 1, 0, ...]
Meaning:    Select features 0, 2, 5, 6
```

### Multi-Objective

- **Maximize** accuracy
- **Minimize** number of features

In [ ]:
if sklearn_available:
    print("="*70)
    print("Feature Selection with Genetic Algorithm")
    print("="*70)

    # Generate data with redundant features
    X_train, y_train, X_val, y_val = generate_synthetic_ml_data(
        n_samples=400, n_features=30, n_informative=10
    )

    print(f"\nTotal features: {X_train.shape[1]}")
    print(f"Informative features: 10")
    print(f"Redundant features: {X_train.shape[1] - 10}")

    # Feature selection GA
    def feature_selection_ga(X_train, y_train, X_val, y_val,
                            pop_size=30, max_generations=50):
        n_features = X_train.shape[1]
        population = initialize_population_binary(pop_size, n_features)

        history = {'n_features': [], 'accuracy': [], 'best': []}
        model = DecisionTreeClassifier(random_state=42, max_depth=5)

        for gen in range(max_generations):
            fitness_values = np.array([
                evaluate_feature_subset(ind, X_train, y_train, X_val, y_val, model)
                for ind in population
            ])

            best_idx = np.argmax(fitness_values)
            best_mask = population[best_idx]
            n_selected = np.sum(best_mask)

            # Calculate actual accuracy
            selected = decode_feature_mask(best_mask)
            if len(selected) > 0:
                model.fit(X_train[:, selected], y_train)
                acc = model.score(X_val[:, selected], y_val)
            else:
                acc = 0.0

            history['n_features'].append(n_selected)
            history['accuracy'].append(acc)
            history['best'].append(fitness_values[best_idx])

            if gen % 10 == 0 or gen == max_generations - 1:
                print(f"Gen {gen:2d} | Features: {n_selected:2d}/{n_features} | "
                      f"Accuracy: {acc:.4f} | Fitness: {fitness_values[best_idx]:.4f}")

            # Evolution
            parents = tournament_selection(population, fitness_values, pop_size)

            offspring = []
            for i in range(0, pop_size, 2):
                p1, p2 = parents[i], parents[min(i+1, pop_size-1)]
                c1, c2 = uniform_crossover(p1, p2, crossover_rate=0.5)
                c1 = bit_flip_mutation(c1, mutation_rate=0.05)
                c2 = bit_flip_mutation(c2, mutation_rate=0.05)
                offspring.extend([c1, c2])

            population = np.array(offspring[:pop_size])

        return best_mask, history

    print("\n" + "-"*70)
    print("Running Feature Selection...")
    print("-"*70 + "\n")

    best_features, fs_history = feature_selection_ga(
        X_train, y_train, X_val, y_val,
        pop_size=30, max_generations=50
    )

    selected_indices = decode_feature_mask(best_features)

    print(f"\n{'='*70}")
    print("FINAL RESULT")
    print(f"{'='*70}")
    print(f"Selected {len(selected_indices)}/{len(best_features)} features")
    print(f"Selected indices: {selected_indices[:10]}..." if len(selected_indices) > 10 else f"Selected indices: {selected_indices}")
    print(f"Final accuracy: {fs_history['accuracy'][-1]:.4f}")

    # Plot evolution
    plot_feature_selection_evolution(fs_history)

else:
    print("⚠️ Skipping feature selection (scikit-learn required)")

## Application 3: Job Scheduling

### Problem

Assign **N jobs** to **M machines** to minimize completion time (makespan).

**Example:**
- 10 jobs with different processing times
- 3 machines
- Goal: Balance load across machines

### GA Encoding

```
Chromosome: [2, 0, 1, 2, 1, 0, ...]
Meaning: Job 0→Machine 2, Job 1→Machine 0, ...
```

In [ ]:
print("="*70)
print("Job Scheduling Optimization")
print("="*70)

# Generate scheduling problem
processing_times, n_machines = generate_scheduling_problem(n_jobs=15, n_machines=3)

print(f"\nNumber of jobs: {len(processing_times)}")
print(f"Number of machines: {n_machines}")
print(f"Processing times: {processing_times[:10]}...")
print(f"Total processing time: {np.sum(processing_times)}")
print(f"Ideal makespan (perfect balance): {np.sum(processing_times) / n_machines:.1f}")

# Scheduling GA
def scheduling_ga(processing_times, n_machines, pop_size=50, max_generations=100):
    n_jobs = len(processing_times)

    # Initialize population (integer encoding)
    population = [np.random.randint(0, n_machines, size=n_jobs) for _ in range(pop_size)]
    population = np.array(population)

    history = {'best': [], 'average': []}

    for gen in range(max_generations):
        # Evaluate (minimize makespan, so fitness = -makespan)
        makespans = np.array([evaluate_schedule(schedule, processing_times, n_machines)
                             for schedule in population])
        fitness_values = -makespans

        history['best'].append(np.min(makespans))
        history['average'].append(np.mean(makespans))

        if gen % 25 == 0 or gen == max_generations - 1:
            print(f"Gen {gen:3d} | Best makespan: {np.min(makespans):.1f} | Avg: {np.mean(makespans):.1f}")

        # Evolution
        parents_real = tournament_selection(population.astype(float), fitness_values, pop_size)
        parents = parents_real.astype(int)

        offspring = []
        for i in range(0, pop_size, 2):
            p1, p2 = parents[i], parents[min(i+1, pop_size-1)]

            # Single-point crossover (valid for integer encoding)
            cx_point = np.random.randint(1, n_jobs)
            c1 = np.concatenate([p1[:cx_point], p2[cx_point:]])
            c2 = np.concatenate([p2[:cx_point], p1[cx_point:]])

            # Random resetting mutation
            for j in range(n_jobs):
                if np.random.rand() < 0.1:
                    c1[j] = np.random.randint(0, n_machines)
                if np.random.rand() < 0.1:
                    c2[j] = np.random.randint(0, n_machines)

            offspring.extend([c1, c2])

        population = np.array(offspring[:pop_size])

    # Find best
    makespans = np.array([evaluate_schedule(schedule, processing_times, n_machines)
                         for schedule in population])
    best_idx = np.argmin(makespans)

    return population[best_idx], makespans[best_idx], history

print("\n" + "-"*70)
print("Running Scheduling GA...")
print("-"*70 + "\n")

best_schedule, best_makespan, sched_history = scheduling_ga(
    processing_times, n_machines, pop_size=50, max_generations=100
)

print(f"\n{'='*70}")
print("FINAL RESULT")
print(f"{'='*70}")
print(f"Best makespan: {best_makespan:.1f}")
print(f"Ideal makespan: {np.sum(processing_times) / n_machines:.1f}")
print(f"Efficiency: {(np.sum(processing_times) / (n_machines * best_makespan)) * 100:.1f}%")

# Visualize schedule
plot_schedule_gantt(best_schedule, processing_times, n_machines)

# Plot convergence
plt.figure(figsize=(10, 5))
plt.plot(sched_history['best'], label='Best Makespan', linewidth=2)
plt.plot(sched_history['average'], label='Average Makespan', linewidth=1.5, linestyle='--')
plt.axhline(y=np.sum(processing_times)/n_machines, color='red', linestyle=':', label='Ideal')
plt.xlabel('Generation')
plt.ylabel('Makespan')
plt.title('Scheduling GA Convergence')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Application 4: Portfolio Optimization

### Problem

Allocate investments across assets to:
- **Maximize** expected return
- **Minimize** risk (variance)

**Markowitz Portfolio Theory**

### GA Encoding

```
Chromosome: [0.2, 0.3, 0.1, 0.25, 0.15]
Meaning: 20% in asset 1, 30% in asset 2, etc.
Constraint: Sum to 100%
```

In [ ]:
print("="*70)
print("Portfolio Optimization")
print("="*70)

# Generate portfolio data
n_assets = 8
returns, cov_matrix = generate_portfolio_data(n_assets)
asset_names = [f'Asset_{chr(65+i)}' for i in range(n_assets)]

print(f"\nNumber of assets: {n_assets}")
print(f"Expected returns: {returns}")

# Portfolio GA
def portfolio_ga(returns, cov_matrix, pop_size=50, max_generations=100, risk_aversion=0.5):
    n_assets = len(returns)
    population = initialize_population_real(pop_size, n_assets, (0, 1))

    history = {'best_return': [], 'best_risk': [], 'best_fitness': []}

    for gen in range(max_generations):
        # Evaluate
        fitness_values = []
        for individual in population:
            weights = decode_portfolio(individual, n_assets)
            fitness = evaluate_portfolio(weights, returns, cov_matrix, risk_aversion)
            fitness_values.append(fitness)

        fitness_values = np.array(fitness_values)

        # Track best
        best_idx = np.argmax(fitness_values)
        best_weights = decode_portfolio(population[best_idx], n_assets)
        best_return = np.dot(best_weights, returns)
        best_variance = np.dot(best_weights, np.dot(cov_matrix, best_weights))
        best_risk = np.sqrt(best_variance)

        history['best_return'].append(best_return)
        history['best_risk'].append(best_risk)
        history['best_fitness'].append(fitness_values[best_idx])

        if gen % 25 == 0 or gen == max_generations - 1:
            print(f"Gen {gen:3d} | Return: {best_return:.4f} | Risk: {best_risk:.4f} | Fitness: {fitness_values[best_idx]:.4f}")

        # Evolution
        parents = rank_based_selection(population, fitness_values, pop_size)

        offspring = []
        for i in range(0, pop_size, 2):
            c1, c2 = simulated_binary_crossover(parents[i], parents[min(i+1, pop_size-1)], eta=20, bounds=(0, 1))
            c1 = polynomial_mutation(c1, 0.1, (0, 1), eta=20)
            c2 = polynomial_mutation(c2, 0.1, (0, 1), eta=20)
            offspring.extend([c1, c2])

        population = np.array(offspring[:pop_size])

    return best_weights, history

print("\n" + "-"*70)
print("Running Portfolio Optimization...")
print("-"*70 + "\n")

best_weights, port_history = portfolio_ga(
    returns, cov_matrix, pop_size=50, max_generations=100, risk_aversion=0.5
)

print(f"\n{'='*70}")
print("FINAL RESULT")
print(f"{'='*70}")
print(f"Optimal weights: {best_weights}")
print(f"Expected return: {port_history['best_return'][-1]:.4f}")
print(f"Risk (std dev): {port_history['best_risk'][-1]:.4f}")

# Visualize
plot_portfolio_weights(best_weights, asset_names)

## Conclusion

### What You've Learned

✅ **Hyperparameter Optimization**
- GA > grid/random search
- Encoding hyperparameters
- Model evaluation as fitness

✅ **Feature Selection**
- Binary encoding for features
- Multi-objective (accuracy vs complexity)
- Reducing overfitting

✅ **Job Scheduling**
- Resource allocation
- Makespan minimization
- Load balancing

✅ **Portfolio Optimization**
- Risk-return tradeoff
- Constraint handling (weights sum to 1)
- Financial applications

### Key Takeaways

1. **GAs are versatile** - applicable to diverse problems
2. **Encoding is crucial** - match representation to problem
3. **Fitness design matters** - include constraints and objectives
4. **Hybrid approaches** - combine with domain knowledge
5. **Practical value** - real performance improvements

### When to Use GAs in Practice

✅ **Good candidates:**
- Complex search spaces
- No gradient information
- Multiple objectives
- Combinatorial problems
- Expensive evaluations (but not too expensive)

❌ **Better alternatives:**
- Simple convex optimization → gradient descent
- Very expensive evaluations → Bayesian optimization
- Need exact optimum → specialized algorithms

### Further Applications

- **Engineering**: Design optimization, circuit design
- **Logistics**: Vehicle routing, warehouse layout
- **AI/ML**: Neural architecture search, AutoML
- **Finance**: Trading strategies, risk management
- **Healthcare**: Treatment planning, resource allocation
- **Manufacturing**: Production scheduling, quality optimization

---

**Congratulations!** You've completed the entire GA tutorial series! 🎉

You now have the knowledge and tools to apply genetic algorithms to real-world problems effectively!

### Next Steps

1. Apply GAs to your own problems
2. Explore specialized variants (CMA-ES, NEAT, etc.)
3. Combine with other techniques (deep learning, RL)
4. Contribute to open-source GA libraries
5. Share your results and learn from the community!

**Happy optimizing!** 🚀